In [1]:
# Импорты и путь к уже сохранённой модели (ничего не обучаем)
import re
import string
from pathlib import Path
from datetime import date, datetime
from random import SystemRandom

import torch
import pymorphy3
from faker import Faker
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline

# Папка с обученной RuBERT должна содержать config.json и веса.
# Если модель в другом месте, укажите полный путь, например:
# MODEL_DIR = Path(r"D:\projects\pii\rubert_pii_fio_extra")
MODEL_DIR = Path("rubert_pii_fio_extra").resolve()

print("Python-окружение использует PyTorch:", torch.__version__)
print("Устройство:", "CUDA" if torch.cuda.is_available() else "CPU")
print("Каталог модели:", MODEL_DIR)

Python-окружение использует PyTorch: 2.14.1+cpu
Устройство: CPU
Каталог модели: C:\Users\Vlad\1JupyterLab\CyberSec\rubert_pii_fio_extra


In [2]:
#Регулярки
# Разрешённые региональные коды серии российского паспорта (лекция 2, слайд 6).
PASSPORT_REGION_RE = r"(?:01|03|04|05|07|08|10|11|12|14|15|17|18|19|20|21|22|24|25|26|27|28|29|32|33|34|35|36|37|38|40|41|42|43|44|45|46|47|48|49|50|51|52|53|54|55|56|57|58|59|60|61|62|63|64|65|66|67|68|69|70|71|72|73|74|75|76|77|78|79|80|81|82|83|84|85|86|87|88|89|90|91|92|93|94|95|96|97|98|99)"
STRUCTURED_TYPES = ("EMAIL","PHONE","INN","CARD","SNILS","BIRTH_DATE",
                    "PASSPORT","URL","MARRIAGE_CERT","CADASTRAL")
PATTERNS = {
    "EMAIL": re.compile( # [!a1_.@] [буквы+спецсимволы] [@] [буквы,цифры,дефисы] [.] [минимум две буквы-цифры] [!a1_+]
        r"(?<![\w.@])[\w.%+\-]+@(?:[A-Za-z0-9\-]+\.)+[A-Za-z]{2,}(?![\w@-])", re.I),
    "PHONE": re.compile(
        r"(?<![\w+])(?:\+7|8)[ \t-]?\(?[3489]\d{2}\)?"   # [!a1_+] [+7|8] [ |/t|-] [(] [3|4|8|9] [xx] [)]
        r"[ \t-]?\d{3}[ \t-]?\d{2}[ \t-]?\d{2}(?![\w-])" # [ |/t|-] [xxx] [ |/t|-] [xx] [ |/t|-] [xx] [!a1_+]
    ),                                                   # неподдерж вне-рф, и "79231116145" 
    "INN": re.compile(r"(?<!\d)\d{12}(?!\d)"),  #[!1][xxxxxxxxxxxx][!1]
    "CARD": re.compile(r"(?<!\d)\d(?:[ -]?\d){12,18}(?![\d-])"),  #visa 13,16,19, Mastercard 16, AMEX 15, MIR 16-19
    "SNILS": re.compile(r"(?<!\d)(?:\d{3}[- ]\d{3}[- ]\d{3}[- ]\d{2}|\d{11})(?!\d)"),  #[!1] [xxx] [ |-] [xxx] [ |-] [xxx] [ |-] [xx] [!1]
    "BIRTH_DATE": re.compile(r"(?<!\d)(?:\d{2}[./-]\d{2}[./-]\d{4}|\d{4}-\d{2}-\d{2})(?!\d)"),
    # Дополнительные четыре обязательных типа нечётного варианта КР
    "PASSPORT": re.compile(r"(?<![0-9])" + PASSPORT_REGION_RE + r"(?:[0-9]{2}|[ \t][0-9]{2})[ \t][0-9]{6}(?![0-9])"),
    "URL": re.compile(r"(?<![\w@])(?:https?://|www\.)[^\s<>\"'«»]+", re.I),
    "MARRIAGE_CERT": re.compile(
        r"(?<!\w)[IVXLCDM]{1,8}[ \t]*[-–][ \t]*[А-ЯЁ]{2}[ \t]*(?:№[ \t]*)?[0-9]{6}(?![0-9])", re.I
    ),
    "CADASTRAL": re.compile(r"(?<![0-9:])[0-9]{2}:[0-9]{2}:[0-9]{6,7}:[0-9]+(?![0-9:])"),
}

In [3]:
#Контрольные суммы
#Удаляет все символы, кроме цифр.
def digits_only(value: str) -> str:
    return re.sub(r"\D", "", value)

#Проверка контрольной суммы алгоритмом Луна.
#1. Цифры проверяемой последовательности нумеруются справа налево.
#2. Цифры, оказавшиеся на нечётных местах, остаются без изменений.
#3. Цифры, стоящие на чётных местах, умножаются на 2.
#4. Если в результате такого умножения возникает число больше 9, 
    #оно заменяется суммой цифр получившегося произведения — однозначным числом, то есть цифрой.
#5. Все полученные в результате преобразования цифры складываются. Если сумма кратна 10, то исходные данные верны.
def valid_luhn(value: str) -> bool:
    number = digits_only(value)
    if (not number.isascii() or not number.isdigit() or not 13 <= len(number) <= 19):
        return False
    total = 0

    for index, char in enumerate(reversed(number)):
        digit = int(char)
        if index % 2:
            digit *= 2
            if digit > 9:
                digit -= 9
        total += digit
    return total % 10 == 0

#Проверка номера банковской карты.
def valid_card(value: str) -> bool:
    number = digits_only(value)

    return (
        13 <= len(number) <= 19  # ну вдруг
        and number[0] in "23456" # 2 мир, мастеркард, 3 amex, 4 visa, 5 mastercard, 6 discover, unionpay
        and len(set(number)) > 1
        and valid_luhn(number)
    )

#Проверка 12-значного ИНН физического лица.
def valid_inn(value: str) -> bool:
    if not re.fullmatch(r"[0-9]{12}", value):
        return False
    numbers = list(map(int, value))
    w11 = (7, 2, 4, 10, 3, 5, 9, 4, 6, 8)    #веса для расчёта контрольной суммы
    w12 = (3, 7, 2, 4, 10, 3, 5, 9, 4, 6, 8)
    c11 = sum(n * w for n, w in zip(numbers[:10], w11)) % 11 % 10
    c12 = sum(n * w for n, w in zip(numbers[:11], w12)) % 11 % 10
    return (c11, c12) == (numbers[10], numbers[11])

#Проверка контрольного числа СНИЛС.
def valid_snils(value: str) -> bool:
    number = digits_only(value)
    if (len(number) != 11 or not number.isascii() or not number.isdigit()):
        return False
    if int(number[:9]) <= 1001998:   # снилс менее 001-001-998 считается по другому, отметаем
        return False

    weighted_sum = sum(int(c) * (9 - i) for i, c in enumerate(number[:9]))
    control = weighted_sum % 101
    if control == 100:
        control = 0
    return control == int(number[-2:])

#Проверяет корректность календарной даты.
def valid_birth_date(value: str) -> bool:
    formats = ("%d.%m.%Y","%d/%m/%Y","%d-%m-%Y","%Y-%m-%d")
    for fmt in formats:
        try:
            parsed = datetime.strptime(value, fmt).date()
            return (
                date(1900, 1, 1)
                <= parsed
                <= date.today()
            )
        except ValueError:
            continue
    return False

In [4]:
#Определяем контекст даты рождения.
def birth_context(text: str, start: int, end: int) -> bool:
    before = text[max(0, start - 60):start]
    after = text[end:min(len(text), end + 20)]

    lead = re.search(
        r"(?:дата\s+рождения|д\.\s*р\.?|"
        r"родил[а-яё]*|день\s+рождения)"
        r"\s*[:\-]?\s*$",
        before,
        re.I
    )

    trail = re.match(
        r"\s*(?:г\.?\s*р\.?|года\s+рождения)"
        r"(?![а-яё\w])",
        after,
        re.I
    )

    return bool(lead or trail)

In [5]:
# Структурированные ПДн
def detect_structured(text: str) -> list[list]:
    if not isinstance(text, str):
        raise TypeError("text должен быть строкой")

    validators = {
        "INN": valid_inn,
        "CARD": valid_card,
        "SNILS": valid_snils,
        "BIRTH_DATE": valid_birth_date
    }

    priority = {
        "EMAIL": 6,
        "PHONE": 5,
        "CARD": 4,
        "INN": 3,
        "SNILS": 2,
        "BIRTH_DATE": 1,
        "PASSPORT": 7,
        "CADASTRAL": 8,
        "MARRIAGE_CERT": 9,
        "URL": 10
    }

    candidates = []
    for label, pattern in PATTERNS.items():
        for match in pattern.finditer(text):
            start, end = match.span()
            value = match.group()

            if label == "URL":
                # Окончание предложения не входит в адрес
                value = value.rstrip(".,;!?)]}")
                end = start + len(value)
                if not valid_course_url(value):
                    continue

            if label == "PASSPORT":
                # Слабая форма с одним пробелом требует явного контекста
                before = text[max(0, start - 45):start]
                if (" " not in value[:5] and "\t" not in value[:5]
                        and not re.search(r"паспорт\w*\s*(?:№|номер|серия|[:№-])?\s*$", before, re.I)):
                    continue

            if label == "MARRIAGE_CERT":
                # У свидетельства о рождении такая же серия; отличаем по словам вокруг
                before = text[max(0, start - 140):start]
                if not re.search(
                    r"свидетельств\w*\s+о\s+(?:заключении\s+)?брак\w*",
                    before, re.I
                ):
                    continue

            # Проверка контрольных разрядов и даты
            if label in validators:
                if not validators[label](value):
                    continue

            # Для СНИЛС без разделителей
            # требуем явное указание типа
            if label == "SNILS" and value.isdigit():
                preceding = text[max(0, start - 25):start]

                if not re.search(
                    r"снилс\s*[:№-]?\s*$",
                    preceding,
                    re.I
                ):
                    continue

            # Дата должна быть датой рождения
            if label == "BIRTH_DATE":
                if not birth_context(text, start, end):
                    continue

            # Проверка длины российского телефона
            if label == "PHONE":
                if len(digits_only(value)) != 11:
                    continue

            candidates.append((label, start, end))

    # Сортировка по приоритету
    candidates.sort(
        key=lambda x: (
            -priority[x[0]],
            -(x[2] - x[1]),
            x[1]
        )
    )

    # Исключение пересекающихся сущностей
    selected = []

    for label, start, end in candidates:
        overlap = any(
            start < e and s < end
            for _, s, e in selected
        )

        if not overlap:
            selected.append((label, start, end))

    # Сортировка по позиции в тексте
    selected.sort(key=lambda x: (x[1], x[2]))

    return [
        [label, start, end]
        for label, start, end in selected
    ]

In [6]:
# Загрузка уже обученной модели из Lab2_run_all.ipynb (без Trainer/train)
_model_config = MODEL_DIR / "config.json"
_model_weights = (
    "model.safetensors", "pytorch_model.bin",
    "model.safetensors.index.json", "pytorch_model.bin.index.json"
)
if not _model_config.is_file() or not any((MODEL_DIR / f).is_file() for f in _model_weights):
    raise FileNotFoundError(
        f"Сохранённая RuBERT не найдена в {MODEL_DIR}. "
        "Положите ноутбук рядом с папкой rubert_pii_fio_extra "
        "или измените MODEL_DIR в первой ячейке."
    )

morph = pymorphy3.MorphAnalyzer()
new_tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR), use_fast=True, local_files_only=True)
new_tokenizer.model_max_length = 128
new_model = AutoModelForTokenClassification.from_pretrained(str(MODEL_DIR), local_files_only=True)
new_ner = pipeline(
    "token-classification",
    model=new_model,
    tokenizer=new_tokenizer,
    aggregation_strategy="first",
    stride=32,
    device=0 if torch.cuda.is_available() else -1
)
print("RuBERT загружена с диска. Обучение не выполнялось.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RuBERT загружена с диска. Обучение не выполнялось.


In [7]:
# Распознавание только ФИО

# Обращения и титулы, не являющиеся именами
PERSON_TITLES = {
    "миссис", "мисс", "мистер",
    "господин", "госпожа",
    "мадам", "сэр"
}

def detect_person(text):
    entities = []

    for entity in new_ner(text):
        label = entity["entity_group"]
        start = entity["start"]
        end = entity["end"]

        value = text[start:end].strip()
        if not any(ch.isalpha() for ch in value):
            continue
        normalized = value.lower()
        # Отсекаем очевидные нарицательные существительные,
        # ошибочно распознанные как NAME или SURNAME.
        if (
            label in {"NAME", "SURNAME"}
            and re.fullmatch(r"[а-яё]+", value)
        ):
            parses = morph.parse(value)

            if (
                any("NOUN" in p.tag for p in parses)
                and not any(
                    "Name" in p.tag or "Surn" in p.tag
                    for p in parses
                )
            ):
                continue
        # Исключаем обращения из распознанных ФИО
        if normalized in PERSON_TITLES:
            continue

        # Отсекаем конкретное ложное срабатывание
        # Но разрешаем «Цыган» с заглавной буквы
        # как потенциальную фамилию
        if label == "NAME" and value == "цыган":
            continue

        entities.append([label, start, end])

    return entities


# Распознавание всех ПДн
def detect_pii(text):
    if not isinstance(text, str):
        raise TypeError("text должен быть строкой")

    result = detect_structured(text)

    for label, start, end in detect_person(text):
        if not any(start < e and s < end for _, s, e in result):
            result.append([label, start, end])

    return sorted(result, key=lambda x: x[1])

In [8]:
# ЛР2: необходимые объекты без запуска обучения RuBERT
import re
import string
import pymorphy3
from faker import Faker
from random import SystemRandom

PERSON_TAGS = {"NAME": "Name", "SURNAME": "Surn"}
morph = pymorphy3.MorphAnalyzer()
anon_fake = Faker("ru_RU")
anon_rng = SystemRandom()

In [9]:
# Фамилии, не изменяющиеся по падежам
INDECLINABLE_SURNAMES = [
    "Гюго", "Дюма", "Верди", "Моро", "Россини", "Моне"
]

INITIAL_LETTERS = "АБВГДЕЁЖЗИЙКЛМНОПРСТУФХЦЧШЩЭЮЯ"


def restore_person_case(original, replacement):
    if original.isupper():
        return replacement.upper()
    if original.islower():
        return replacement.lower()
    if original.istitle():
        return replacement.capitalize()
    return replacement


def replace_person_base(value, label, replacements, left_context=""):
    if label not in PERSON_TAGS or not value:
        raise ValueError(f"Некорректный атрибут: {label}, {value!r}")

    # 1. Инициалы: М -> К, М. -> К.
    initial = re.fullmatch(r"([А-Яа-яЁё])(\.?)", value)

    if initial:
        letter, dot = initial.groups()
        key = (label, "INITIAL", letter.upper())

        if key not in replacements:
            replacements[key] = anon_rng.choice([
                c for c in INITIAL_LETTERS
                if c != letter.upper()
            ])

        return restore_person_case(
            letter, replacements[key]
        ) + dot

    # 2. Обычный морфологический анализ
    tag = PERSON_TAGS[label]
    parses = [
        p for p in morph.parse(value)
        if tag in p.tag
        and p.tag.gender in ("masc", "femn")
    ]

    # Запасной анализ редких имён:
    # Робера, Фауста, Эдисона и т.п.
    if not parses and label == "NAME":
        parses = [
            p for p in morph.parse(value)
            if "NOUN" in p.tag
            and "anim" in p.tag
            and p.tag.gender in ("masc", "femn")
            and p.tag.case is not None
        ]

    if parses:
        original = parses[0]
        gender = original.tag.gender
        case = original.tag.case
        number = original.tag.number
        normal_form = original.normal_form

    elif label == "SURNAME":
        # 3. Неизвестная склоняемая фамилия:
        # Брохесом -> предположительно мужской, тв. падеж
        if value.lower().endswith(("ом", "ем")):
            gender = "masc"
            case = "ablt"
            number = "sing"
            normal_form = value[:-2].lower()

        else:
            # 4. Неоднозначная или несклоняемая фамилия.
            # Используем другую несклоняемую фамилию:
            # Мураками -> Верди, Моне -> Гюго.
            key = ("SURNAME", "INDECL", value.lower())

            if key not in replacements:
                candidates = [
                    s for s in INDECLINABLE_SURNAMES
                    if s.lower() != value.lower()
                ]
                replacements[key] = anon_rng.choice(candidates)

            return restore_person_case(
                value, replacements[key]
            )


    elif label == "NAME":
        lower = value.lower()

        if lower == "лизка":
            gender = "femn"
            case = "nomn"
            number = "sing"
            normal_form = "лизка"

        elif (
            lower == "мая"
            and re.search(r"\bмальчика\s+$", left_context.lower())
        ):
            gender = "masc"
            case = "gent"
            number = "sing"
            normal_form = "май"

        elif lower.endswith(("ом", "ем")):
            # Орсоном -> Орсон, творительный падеж
            gender = "masc"
            case = "ablt"
            number = "sing"
            normal_form = lower[:-2]

        elif lower.endswith(("ей", "ий", "й")):
            # Кащей -> Кащей, предполагаемый именительный
            gender = "masc"
            case = "nomn"
            number = "sing"
            normal_form = lower

        else:
            raise ValueError(
                f"Неизвестная форма NAME: {value}"
            )

    # 5. Согласование повторов по нормальной форме
    key = (label, normal_form, gender)
    grammemes = {g for g in (case, number) if g}

    if key not in replacements:
        sex = "male" if gender == "masc" else "female"
        method = "first_name" if label == "NAME" else "last_name"

        for _ in range(100):
            candidate = getattr(
                anon_fake, f"{method}_{sex}"
            )()

            options = [
                p for p in morph.parse(candidate)
                if tag in p.tag and p.tag.gender == gender
            ]

            if not options:
                continue

            candidate_parse = options[0]

            if candidate_parse.normal_form == normal_form:
                continue

            if grammemes and candidate_parse.inflect(grammemes) is None:
                continue

            replacements[key] = candidate_parse
            break
        else:
            raise ValueError(
                f"Не удалось подобрать замену: {value}"
            )

    # 6. Склонение выбранной замены
    replacement = replacements[key]

    inflected = (
        replacement.inflect(grammemes)
        if grammemes else replacement
    )

    if inflected is None:
        raise ValueError(f"Ошибка склонения: {value}")

    return restore_person_case(value, inflected.word)

In [10]:
# Расширение базового алгоритма: уменьшительные имена и латиница

latin_fake = Faker("en_US")

LATIN_FEMALE = {"bonnie"}
LATIN_MALE = {"george", "walt"}


def replace_person(value, label, replacements, left_context=""):
    # Неоднозначное по роду имя: Саша
    if label == "NAME" and value.casefold() == "саша":
        key = ("NAME", "NICKNAME", "саша")

        if key not in replacements:
            # Женя и Валя могут быть мужскими и женскими именами
            replacements[key] = anon_rng.choice(["Женя", "Валя"])

        return restore_person_case(value, replacements[key])

    # Латинский инициал: H -> R
    if label == "NAME" and re.fullmatch(r"[A-Za-z]", value):
        key = ("NAME", "LATIN_INITIAL", value.upper())

        if key not in replacements:
            letters = string.ascii_uppercase.replace(value.upper(), "")
            replacements[key] = anon_rng.choice(letters)

        return restore_person_case(value, replacements[key])

    # Латинские имена: George -> Michael
    if label == "NAME" and re.fullmatch(r"[A-Za-z]+", value):
        key = ("NAME", "LATIN", value.casefold())

        if key not in replacements:
            name = value.casefold()

            if name in LATIN_FEMALE:
                generate = latin_fake.first_name_female
            elif name in LATIN_MALE:
                generate = latin_fake.first_name_male
            else:
                # Пол неизвестен: нейтральные варианты
                generate = lambda: anon_rng.choice(
                    ["Alex", "Taylor", "Jordan", "Casey"]
                )

            for _ in range(100):
                candidate = generate()

                if candidate.casefold() != name:
                    replacements[key] = candidate
                    break
            else:
                raise ValueError(f"Не удалось заменить NAME: {value}")

        return restore_person_case(value, replacements[key])

    # Все обычные кириллические имена и фамилии
    return replace_person_base(
        value, label, replacements, left_context
    )

In [11]:
import re
import string
from random import SystemRandom

anon_rng = SystemRandom()


def random_other(char, alphabet):
    """Случайный символ, обязательно отличный от исходного."""
    return anon_rng.choice(alphabet.replace(char, ""))


def replace_email(value):
    """Меняет часть до @, сохраняя домен и оформление."""
    parts = value.split("@")

    if len(parts) != 2:
        raise ValueError("Некорректный EMAIL")

    local, domain = parts

    # Проверка некорректного расположения точек и дефисов
    if (
        local.startswith(".")
        or local.endswith(".")
        or ".." in local
        or ".." in domain
        or any(
            part.startswith("-") or part.endswith("-")
            for part in domain.split(".")
        )
    ):
        raise ValueError(f"Некорректный EMAIL: {value}")

    if not re.fullmatch(r"[A-Za-z0-9._%+-]+", local):
        raise ValueError("Некорректная локальная часть EMAIL")

    if not re.fullmatch(
        r"[A-Za-zА-Яа-яЁё0-9-]+(?:\.[A-Za-zА-Яа-яЁё0-9-]+)+",
        domain
    ):
        raise ValueError("Некорректный домен EMAIL")

    result = []

    for char in local:
        if char in string.ascii_lowercase:
            result.append(random_other(char, string.ascii_lowercase))
        elif char in string.ascii_uppercase:
            result.append(random_other(char, string.ascii_uppercase))
        elif char in string.digits:
            result.append(random_other(char, string.digits))
        else:
            result.append(char)

    return "".join(result) + "@" + domain


def replace_phone(value):
    """Сохраняет первые 4 цифры и все нецифровые символы."""
    digits = [c for c in value if c in string.digits]

    if not 10 <= len(digits) <= 15:
        raise ValueError("Некорректная длина PHONE")

    if any(c not in string.digits + "+-(). \t" for c in value):
        raise ValueError("Недопустимые символы в PHONE")

    result = []
    digit_index = 0

    for char in value:
        if char in string.digits:
            if digit_index < 4:
                result.append(char)
            else:
                result.append(random_other(char, string.digits))
            digit_index += 1
        else:
            result.append(char)

    return "".join(result)

In [12]:
def replace_inn(value):
    """Сохраняет первые 4 цифры и пересчитывает контрольные разряды."""

    if not valid_inn(value):
        raise ValueError(f"Некорректный ИНН: {value}")

    w11 = (7, 2, 4, 10, 3, 5, 9, 4, 6, 8)
    w12 = (3, 7, 2, 4, 10, 3, 5, 9, 4, 6, 8)

    while True:
        first10 = value[:4] + "".join(
            str(anon_rng.randrange(10)) for _ in range(6)
        )

        numbers = list(map(int, first10))

        c11 = sum(
            n * w for n, w in zip(numbers, w11)
        ) % 11 % 10

        c12 = sum(
            n * w for n, w in zip(numbers + [c11], w12)
        ) % 11 % 10

        result = first10 + str(c11) + str(c12)

        if result != value:
            return result

In [13]:
def replace_card(value):
    """Сохраняет первые 6 и последние 4 цифры, формат и Luhn."""

    if (
        not re.fullmatch(r"[0-9]+(?:[ \t-][0-9]+)*", value)
        or not valid_card(value)
    ):
        raise ValueError(f"Некорректная банковская карта: {value}")

    original = digits_only(value)

    for _ in range(1000):
        middle = "".join(
            str(anon_rng.randrange(10))
            for _ in range(len(original) - 10)
        )

        candidate = original[:6] + middle + original[-4:]

        if candidate != original and valid_luhn(candidate):
            digits_iter = iter(candidate)

            return "".join(
                next(digits_iter) if char in "0123456789" else char
                for char in value
            )

    raise RuntimeError("Не удалось сгенерировать валидную карту")

In [14]:
# Дополнительные валидаторы и генераторы для нечётного варианта курсовой
# Используют уже созданные valid_snils(), digits_only(), restore_digits(), anon_rng.
from urllib.parse import urlsplit, urlunsplit, parse_qsl, urlencode

COURSE_TYPES = frozenset({
    "NAME", "SURNAME", "INN", "PASSPORT", "CARD", "SNILS",
    "URL", "PHONE", "MARRIAGE_CERT", "CADASTRAL"
})
EXTRA_ANON_TYPES = {"PASSPORT", "SNILS", "URL", "MARRIAGE_CERT", "CADASTRAL"}

PASSPORT_FULL = re.compile(PASSPORT_REGION_RE + r"(?:[0-9]{2}|[ \t][0-9]{2})[ \t][0-9]{6}")
MARRIAGE_FULL = re.compile(
    r"[IVXLCDM]{1,8}[ \t]*[-–][ \t]*[А-ЯЁ]{2}[ \t]*(?:№[ \t]*)?[0-9]{6}", re.I
)
CADASTRAL_FULL = re.compile(r"[0-9]{2}:[0-9]{2}:[0-9]{6,7}:[0-9]+")


def random_digits(n):
    return "".join(str(anon_rng.randrange(10)) for _ in range(n))


def valid_course_url(value):
    try:
        parsed = urlsplit(value if re.match(r"https?://", value, re.I)
                          else "https://" + value)
        host = parsed.hostname
        port = parsed.port  # проверяет корректность записи порта
        return (
            parsed.scheme in ("http", "https")
            and bool(host and "." in host)
            and not parsed.username and not parsed.password
            and bool(re.fullmatch(r"[A-Za-z0-9.-]+", host))
            and all(x for x in host.split("."))
            and (port is None or 1 <= port <= 65535)
        )
    except ValueError:
        return False


def replace_passport(value):
    """Паспорт РФ: сохраняем четыре цифры серии, меняем 6 цифр номера."""
    if not PASSPORT_FULL.fullmatch(value):
        raise ValueError(f"Некорректный PASSPORT: {value}")
    digits = digits_only(value)
    while True:
        new_digits = digits[:4] + random_digits(6)
        if new_digits != digits and new_digits[-6:] != "000000":
            return restore_digits(value, new_digits)


def snils_control(body):
    weighted = sum(int(d) * (9 - i) for i, d in enumerate(body))
    value = weighted % 101
    return f"{0 if value == 100 else value:02d}"


def replace_snils(value):
    """Меняем первые 9 цифр СНИЛС; контрольные последние 2 оставляем исходными."""
    s = digits_only(value)
    if not (re.fullmatch(r"[0-9]{11}|[0-9]{3}[- ][0-9]{3}[- ][0-9]{3}[- ][0-9]{2}", value)
            and valid_snils(value)):
        raise ValueError(f"Некорректный SNILS: {value}")

    target_control = s[-2:]
    # Подбираем случайное новое основание, которое даёт исходное контрольное число.
    for _ in range(10000):
        body = str(anon_rng.randrange(1001999, 10**9)).zfill(9)
        if body == s[:9] or snils_control(body) != target_control:
            continue
        candidate = body + target_control
        if valid_snils(candidate):
            return restore_digits(value, candidate)
    raise RuntimeError("Не удалось подобрать СНИЛС с сохранённым контрольным числом")


def replace_marriage_cert(value):
    """Свидетельство о браке: сохраняем серию и знак №, меняем номер."""
    if not MARRIAGE_FULL.fullmatch(value):
        raise ValueError(f"Некорректный MARRIAGE_CERT: {value}")
    while True:
        tail = random_digits(6)
        if tail != value[-6:] and tail != "000000":
            return value[:-6] + tail


def replace_cadastral(value):
    """Сохраняем округ, район, кадастровый квартал и длину номера объекта."""
    if not CADASTRAL_FULL.fullmatch(value):
        raise ValueError(f"Некорректный CADASTRAL: {value}")
    prefix, last = value.rsplit(":", 1)
    while True:
        new = random_digits(len(last))
        if new != last and int(new) != 0:
            return prefix + ":" + new


def replace_url(value):
    """Генерируем синтаксически валидный URL на зарезервированном .example."""
    if not valid_course_url(value):
        raise ValueError(f"Некорректный URL: {value}")
    has_scheme = bool(re.match(r"https?://", value, re.I))
    parts = urlsplit(value if has_scheme else "https://" + value)
    alphabet = string.ascii_lowercase + string.digits
    def token(length=8):
        return "".join(anon_rng.choice(alphabet) for _ in range(max(3, min(length, 16))))
    new_host = ("www." if value.lower().startswith("www.") else "") + "u" + token() + ".example"
    if parts.port:
        new_host += f":{parts.port}"
    new_path = "/".join(token(len(part)) if part else "" for part in parts.path.split("/"))
    pairs = parse_qsl(parts.query, keep_blank_values=True)
    new_query = urlencode([(token(len(k)), token(len(v))) for k, v in pairs]) if pairs else ""
    new_fragment = token(len(parts.fragment)) if parts.fragment else ""
    result = urlunsplit((parts.scheme, new_host, new_path, new_query, new_fragment))
    return result if has_scheme else result[len("https://"):]

NEW_REPLACERS = {
    "PASSPORT": replace_passport,
    "SNILS": replace_snils,
    "URL": replace_url,
    "MARRIAGE_CERT": replace_marriage_cert,
    "CADASTRAL": replace_cadastral,
}


In [15]:
ANON_TYPES = {"NAME", "SURNAME", "EMAIL", "INN", "PHONE", "CARD"} | COURSE_TYPES


def restore_digits(original, new_digits):
    """Возвращает новые цифры в исходное оформление."""
    digits = iter(new_digits)
    return "".join(
        next(digits) if ch in "0123456789" else ch
        for ch in original
    )


def anon(text, entities):
    """Синонимически обезличивает ПДн по координатам из ЛР1."""

    if not isinstance(text, str):
        raise TypeError("Текст должен быть строкой")

    # Допускаем как один атрибут, так и список атрибутов
    if isinstance(entities, (list, tuple)) and len(entities) == 3 and isinstance(entities[0], str):
        entities = [entities]

    spans = []

    for entity in entities:
        if len(entity) != 3:
            raise ValueError(f"Некорректная сущность: {entity}")

        label, start, end = entity

        if not (isinstance(start, int) and isinstance(end, int)
                and 0 <= start < end <= len(text)):
            raise ValueError(f"Некорректные координаты: {entity}")

        if label in ANON_TYPES or label in EXTRA_ANON_TYPES:
            spans.append((label, start, end))

    spans.sort(key=lambda x: x[1])

    for left, right in zip(spans, spans[1:]):
        if left[2] > right[1]:
            raise ValueError("Обнаружены пересекающиеся сущности")

    person_cache = {}
    structured_cache = {}
    result = text

    for label, start, end in reversed(spans):
        original = text[start:end]

        if label in ("NAME", "SURNAME"):
            replacement = replace_person(
                original,
                label,
                person_cache,
                left_context=text[max(0, start - 80):start]
            )

        elif label in EXTRA_ANON_TYPES:
            # Отдельные валидаторы выполняются ДО обращения к кешу
            # (иначе неверный формат может пройти через ранее созданную замену).
            new_value = NEW_REPLACERS[label](original) if (label, original) not in structured_cache else structured_cache[(label, original)]
            structured_cache[(label, original)] = new_value
            replacement = new_value

        elif label == "EMAIL":
            key = (label, original)

            if key not in structured_cache:
                structured_cache[key] = replace_email(original)

            replacement = structured_cache[key]

        else:
            # Нормализуем числовой атрибут
            normalized = digits_only(original)
            key = (label, normalized)

            # Проверяем каждое входное значение
            if label == "INN" and not valid_inn(original):
                raise ValueError(f"Некорректный ИНН: {original}")

            if label == "CARD" and (
                not re.fullmatch(r"[0-9]+(?:[ \t-][0-9]+)*", original)
                or not valid_card(original)
            ):
                raise ValueError(f"Некорректная карта: {original}")

            if label == "PHONE":
                if (
                    not 10 <= len(normalized) <= 15
                    or any(c not in "0123456789+-(). \t" for c in original)
                ):
                    raise ValueError(f"Некорректный телефон: {original}")

            if key not in structured_cache:
                generators = {
                    "INN": replace_inn,
                    "PHONE": replace_phone,
                    "CARD": replace_card
                }

                generated = generators[label](original)
                structured_cache[key] = digits_only(generated)

            replacement = restore_digits(
                original, structured_cache[key]
            )

        if replacement == original:
            raise ValueError(
                f"Значение не изменилось: {label} = {original}"
            )

        # Заменяем справа налево
        result = result[:start] + replacement + result[end:]

    return result

In [16]:
text = (
    "Иванов Иван написал с ivan.petrov1988@gmail.com, "
    "указал ИНН 770698959438, телефон +7 (999) 123-45-67 "
    "и карту 4561 2612 1234 5467. "
    "Позднее с Ивановым Иваном связались повторно."
)

samples = [
    ("SURNAME", "Иванов"),
    ("NAME", "Иван"),
    ("EMAIL", "ivan.petrov1988@gmail.com"),
    ("INN", "770698959438"),
    ("PHONE", "+7 (999) 123-45-67"),
    ("CARD", "4561 2612 1234 5467"),
    ("SURNAME", "Ивановым"),
    ("NAME", "Иваном")
]

entities = []

for label, value in samples:
    match = re.search(
        rf"(?<!\w){re.escape(value)}(?!\w)", text
    )
    assert match is not None, f"Не найдено: {value}"
    entities.append([label, match.start(), match.end()])

first = anon(text, entities)
second = anon(text, entities)

print("ИСХОДНЫЙ ТЕКСТ:")
print(text)

print("\nОБЕЗЛИЧЕННЫЙ ТЕКСТ:")
print(first)

print("\nПОВТОРНЫЙ ЗАПУСК:")
print(second)

assert first != text
assert first != second

print("\nПроверка пройдена.")

ИСХОДНЫЙ ТЕКСТ:
Иванов Иван написал с ivan.petrov1988@gmail.com, указал ИНН 770698959438, телефон +7 (999) 123-45-67 и карту 4561 2612 1234 5467. Позднее с Ивановым Иваном связались повторно.

ОБЕЗЛИЧЕННЫЙ ТЕКСТ:
Данилов Христофор написал с ytls.dtutmh2272@gmail.com, указал ИНН 770646252775, телефон +7 (999) 262-34-76 и карту 4561 2660 9519 5467. Позднее с Даниловым Христофором связались повторно.

ПОВТОРНЫЙ ЗАПУСК:
Михеев Терентий написал с vbsc.auylqz2729@gmail.com, указал ИНН 770693398806, телефон +7 (999) 084-30-89 и карту 4561 2655 6959 5467. Позднее с Михеевым Терентием связались повторно.

Проверка пройдена.


In [17]:
# Расчётные проверки требований ЛР2: формулы с реальными цифрами,
# полученными при конкретном вызове anon(). Проверки assert сохранены.
from IPython.display import display, Math


def formula(latex):
    display(Math(latex))


def written(value):
    """Отображение тестового значения в математическом выводе."""
    return r"\texttt{" + value.replace(" ", r"\ ") + "}"


def inn_control_equation(number, digit_count, weights, index):
    """Подробное вычисление 11-го или 12-го разряда ИНН."""
    numbers = [int(n) for n in number[:digit_count]]
    products = [n * w for n, w in zip(numbers, weights)]
    weighted_sum = sum(products)
    control = weighted_sum % 11 % 10
    terms = " + ".join(rf"{n}\cdot {w}" for n, w in zip(numbers, weights))
    formula(
        rf"c_{{{index}}} = "
        rf"\bigl(( {terms} )\bmod 11\bigr)\bmod 10"
        rf" = ({weighted_sum}\bmod 11)\bmod 10"
        rf" = {weighted_sum % 11}\bmod 10 = {control} = d_{{{index}}}"
    )
    assert control == int(number[index-1]), f"Неверный разряд ИНН {index}"


def luhn_equation(number):
    """Числовая подстановка в формулу Луна, начиная с правого края."""
    transformed = []
    terms = []
    for index, ch in enumerate(reversed(number)):
        n = int(ch)
        if index % 2:
            adjusted = 2 * n - 9 if 2 * n > 9 else 2 * n
            terms.append(rf"(2\cdot {n}" + ("-9)" if 2*n > 9 else ")"))
        else:
            adjusted = n
            terms.append(str(n))
        transformed.append(adjusted)
    total = sum(transformed)
    formula(rf"S_{{\mathrm{{Luhn}}}} = {' + '.join(terms)} = {total}")
    formula(rf"S_{{\mathrm{{Luhn}}}}\bmod 10 = {total}\bmod 10 = {total%10} = 0")
    assert total % 10 == 0


def format_equations(original, replacement):
    """Проверка сохранения индексов и значений всех нецифровых знаков."""
    left = [(i, ord(c)) for i, c in enumerate(original) if c not in '0123456789']
    right = [(i, ord(c)) for i, c in enumerate(replacement) if c not in '0123456789']
    # Код символа показан как Unicode-код, чтобы пробелы были видны и проверяемы.
    to_tex = lambda pairs: r"\left[" + r",\ ".join(f"({pos},{code})" for pos, code in pairs) + r"\right]"
    formula(
        r"\operatorname{nonDigits}_{(\mathrm{index},\mathrm{Unicode})}(x)=" + to_tex(left)
    )
    formula(
        r"\operatorname{nonDigits}_{(\mathrm{index},\mathrm{Unicode})}(y)=" + to_tex(right)
    )
    formula(rf"\lvert \operatorname{{nonDigits}}(x)\rvert = {len(left)} = {len(right)} = \lvert \operatorname{{nonDigits}}(y)\rvert")
    assert left == right, f"Нарушено оформление: {original} => {replacement}"


def check_structured_anonymization(label, original, verbose=True):
    replacement = anon(original, [label, 0, len(original)])
    old_digits, new_digits = digits_only(original), digits_only(replacement)

    checks = {
        'изменение значения': replacement != original,
        'число цифр': len(old_digits) == len(new_digits),
    }

    if label == 'INN':
        checks.update({
            'префикс': old_digits[:4] == new_digits[:4],
            'длина 12': len(new_digits) == 12,
            'контрольные цифры': valid_inn(replacement),
        })
    elif label == 'PHONE':
        checks.update({
            'префикс': old_digits[:4] == new_digits[:4],
            'разделители': [(i,c) for i,c in enumerate(original) if c not in '0123456789'] ==
                           [(i,c) for i,c in enumerate(replacement) if c not in '0123456789'],
        })
    elif label == 'CARD':
        checks.update({
            'BIN': old_digits[:6] == new_digits[:6],
            'последние 4': old_digits[-4:] == new_digits[-4:],
            'Luhn': valid_luhn(replacement),
            'формат карты': valid_card(replacement),
            'разделители': [(i,c) for i,c in enumerate(original) if c not in '0123456789'] ==
                           [(i,c) for i,c in enumerate(replacement) if c not in '0123456789'],
        })
    else:
        raise ValueError('Ожидался INN, PHONE или CARD')

    assert all(checks.values()), f'Проверка {label} не пройдена: {checks}'

    if verbose:
        formula(rf"\mathrm{{{label}}}:\quad {written(original)} \longrightarrow {written(replacement)}")
        formula(rf"\lvert \operatorname{{digits}}(x)\rvert = {len(old_digits)} = {len(new_digits)} = \lvert \operatorname{{digits}}(y)\rvert")
        if label == 'INN':
            formula(rf"\operatorname{{prefix}}_4(x) = {old_digits[:4]} = {new_digits[:4]} = \operatorname{{prefix}}_4(y)")
            w11 = (7, 2, 4, 10, 3, 5, 9, 4, 6, 8)
            w12 = (3, 7, 2, 4, 10, 3, 5, 9, 4, 6, 8)
            inn_control_equation(new_digits, 10, w11, 11)
            inn_control_equation(new_digits, 11, w12, 12)

        elif label == 'PHONE':
            formula(rf"\operatorname{{prefix}}_4(x) = {old_digits[:4]} = {new_digits[:4]} = \operatorname{{prefix}}_4(y)")
            format_equations(original, replacement)

        elif label == 'CARD':
            formula(rf"\operatorname{{BIN}}_6(x) = {old_digits[:6]} = {new_digits[:6]} = \operatorname{{BIN}}_6(y)")
            formula(rf"\operatorname{{suffix}}_4(x) = {old_digits[-4:]} = {new_digits[-4:]} = \operatorname{{suffix}}_4(y)")
            formula(rf"13 \leq {len(new_digits)} \leq 19")
            luhn_equation(new_digits)
            format_equations(original, replacement)
        formula(rf"x \neq y \quad ({written(original)} \neq {written(replacement)})")
    return replacement


examples = {
    'INN': '770698959438',
    'PHONE': '+7 (999) 123-45-67',
    'CARD': '4561 2612 1234 5467',
}

for label, original in examples.items():
    assert {
        'INN': lambda s: valid_inn(s),
        'PHONE': lambda s: len(digits_only(s)) == 11,
        'CARD': lambda s: valid_card(s),
    }[label](original)
    check_structured_anonymization(label, original)

# 100 вызовов anon() на атрибут: показываем число различных замен в виде формулы.
for label, original in examples.items():
    results = {check_structured_anonymization(label, original, verbose=False) for _ in range(100)}
    assert len(results) > 1, f'Нет случайности для {label}'
    formula(rf"\mathrm{{{label}}}:\quad \left|\{{\operatorname{{anon}}_1,\ldots,\operatorname{{anon}}_{{100}}\}}\right| = {len(results)} > 1")


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

... (17 outputs left out)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [18]:
# Самопроверка пяти добавленных типов (без обучения / без RuBERT)
examples = {
    "PASSPORT": "45 01 123456",
    "SNILS": "112-233-445-" + snils_control("112233445"),
    "URL": "https://example.org/profile/user?q=7#info",
    "MARRIAGE_CERT": "II-АГ № 123456",
    "CADASTRAL": "77:01:0004012:123",
}
for label, old in examples.items():
    new = anon(old, [label, 0, len(old)])
    assert new != old, label
    if label == "PASSPORT":
        assert digits_only(new)[:4] == digits_only(old)[:4]
        assert PASSPORT_FULL.fullmatch(new)
    elif label == "SNILS":
        assert valid_snils(new)
        assert digits_only(new)[-2:] == digits_only(old)[-2:]
        assert digits_only(new)[:9] != digits_only(old)[:9]
        assert all((a.isdigit() == b.isdigit()) and (a.isdigit() or a == b)
                   for a, b in zip(old, new))
    elif label == "URL":
        assert valid_course_url(new) and urlsplit(new).scheme == "https"
    elif label == "MARRIAGE_CERT":
        assert MARRIAGE_FULL.fullmatch(new) and old[:-6] == new[:-6]
    elif label == "CADASTRAL":
        assert CADASTRAL_FULL.fullmatch(new) and old.rsplit(':', 1)[0] == new.rsplit(':', 1)[0]
    print(f"{label}: {old}  →  {new}")

# Недопустимые серии паспорта не должны находиться.
for bad_passport in ("00 00 123456", "0000 123456", "02 12 123456"):
    assert not PASSPORT_FULL.fullmatch(bad_passport)
    bad_text = "Паспорт " + bad_passport
    assert not any(label == "PASSPORT" for label, _, _ in detect_structured(bad_text))

text_course = (
    "Паспорт 45 01 123456, СНИЛС " + examples["SNILS"] +
    ", сайт https://example.org/profile, "
    "свидетельство о заключении брака II-АГ № 123456, "
    "кадастровый номер 77:01:0004012:123."
)
found = detect_structured(text_course)
for label in examples:
    assert any(x[0] == label for x in found), (label, found)
assert all(text_course[s:e] for _, s, e in found)
print("Автораспознавание добавленных типов:", [(k, text_course[s:e]) for k, s, e in found])
print("Обезличено:", anon(text_course, [x for x in found if x[0] in COURSE_TYPES]))


PASSPORT: 45 01 123456  →  45 01 067005
SNILS: 112-233-445-95  →  876-886-322-95
URL: https://example.org/profile/user?q=7#info  →  https://uvcu7j6nl.example/lsxo7zd/xrjf?bko=5d2#imm8
MARRIAGE_CERT: II-АГ № 123456  →  II-АГ № 982262
CADASTRAL: 77:01:0004012:123  →  77:01:0004012:561
Автораспознавание добавленных типов: [('PASSPORT', '45 01 123456'), ('SNILS', '112-233-445-95'), ('URL', 'https://example.org/profile'), ('MARRIAGE_CERT', 'II-АГ № 123456'), ('CADASTRAL', '77:01:0004012:123')]
Обезличено: Паспорт 45 01 089115, СНИЛС 309-715-863-95, сайт https://u3aue0i2l.example/mkxqni1, свидетельство о заключении брака II-АГ № 444457, кадастровый номер 77:01:0004012:337.


In [19]:
# МЕНЯЙТЕ ТОЛЬКО ЭТИ ДВЕ СТРОКИ ДЛЯ СВОЕГО ПРИМЕРА
TEXT = "Иван получил кредит"
ENTITIES = ["NAME", 0, 4]  # или None для автоматического распознавания

if ENTITIES is None:
    selected_entities = [
        e for e in detect_pii(TEXT)
        if e[0] in COURSE_TYPES
    ]
    print("Режим: автоматический (RuBERT + правила)")
else:
    selected_entities = ENTITIES
    print("Режим: ручная разметка из ЛР1")

items = ([selected_entities] if isinstance(selected_entities, (list, tuple))
         and len(selected_entities) == 3 and isinstance(selected_entities[0], str)
         else selected_entities)
print("Найденные/заданные ПДн:", [
    (label, TEXT[start:end], start, end) for label, start, end in items
])
print("\nБЫЛО:", TEXT)
print("СТАЛО:", anon(TEXT, selected_entities))

Режим: ручная разметка из ЛР1
Найденные/заданные ПДн: [('NAME', 'Иван', 0, 4)]

БЫЛО: Иван получил кредит
СТАЛО: Богдан получил кредит


In [20]:
auto_text = "Иванов Иван, телефон +7 (999) 123-45-67."
auto_entities = [e for e in detect_pii(auto_text) if e[0] in COURSE_TYPES]
print("Автоматически найдены:", [(k, auto_text[s:e]) for k,s,e in auto_entities])
print("Исходный текст:", auto_text)
print("Обезличенный:", anon(auto_text, auto_entities))

Автоматически найдены: [('SURNAME', 'Иванов'), ('NAME', 'Иван'), ('PHONE', '+7 (999) 123-45-67')]
Исходный текст: Иванов Иван, телефон +7 (999) 123-45-67.
Обезличенный: Князев Марк, телефон +7 (999) 604-74-18.


In [21]:

text = (
    "Паспорт 45 01 123456, "
    "кадастровый номер 77:01:0004012:123, "
    "сайт https://example.org/profile."
)

entities = [
    e for e in detect_pii(text)
    if e[0] in COURSE_TYPES
]

print("Найденные ПДн:", [
    (label, text[start:end])
    for label, start, end in entities
])

print("Обезличенный текст:")
print(anon(text, entities))


Найденные ПДн: [('PASSPORT', '45 01 123456'), ('CADASTRAL', '77:01:0004012:123'), ('URL', 'https://example.org/profile')]
Обезличенный текст:
Паспорт 45 01 019195, кадастровый номер 77:01:0004012:520, сайт https://uxubfkrw4.example/iqns613.


In [22]:

import re

# 1. Проверяем распознавание
text = "Выдано свидетельство о заключении брака II-АГ № 123456."

entities = [
    x for x in detect_structured(text)
    if x[0] == "MARRIAGE_CERT"
]

assert len(entities) == 1, "Свидетельство не найдено"

label, start, end = entities[0]
original = text[start:end]

# 2. Обезличиваем
result = anon(text, entities)
replacement = result[start:start + len(original)]

print("НАЙДЕНО:", entities)
print("БЫЛО:", text)
print("СТАЛО:", result)

# 3. Проверяем требования
assert replacement != original
assert replacement[:-6] == original[:-6]
assert replacement[-6:].isdigit()
assert len(replacement[-6:]) == 6
assert MARRIAGE_FULL.fullmatch(replacement)

# 4. Проверяем случайность
variants = {
    anon(text, entities)
    for _ in range(20)
}
assert len(variants) > 1

# 5. Проверяем отличие от свидетельства о рождении
birth_text = "Выдано свидетельство о рождении II-АГ № 123456."

birth_entities = [
    x for x in detect_structured(birth_text)
    if x[0] == "MARRIAGE_CERT"
]
assert birth_entities == []

# 6. Проверяем одинаковую замену повторов
repeat_text = (
    "Свидетельство о заключении брака II-АГ № 123456. "
    "Повторно свидетельство о заключении брака II-АГ № 123456."
)
repeat_entities = [
    x for x in detect_structured(repeat_text)
    if x[0] == "MARRIAGE_CERT"
]

assert len(repeat_entities) == 2
repeat_result = anon(repeat_text, repeat_entities)

s1, e1 = repeat_entities[0][1:]
s2, e2 = repeat_entities[1][1:]

assert repeat_result[s1:e1] == repeat_result[s2:e2]

print("\nВсе проверки свидетельства о браке пройдены.")
print("Уникальных результатов из 20:", len(variants))


НАЙДЕНО: [['MARRIAGE_CERT', 40, 54]]
БЫЛО: Выдано свидетельство о заключении брака II-АГ № 123456.
СТАЛО: Выдано свидетельство о заключении брака II-АГ № 455326.

Все проверки свидетельства о браке пройдены.
Уникальных результатов из 20: 20
